# Qualitative analysis


In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt
from PIL import Image
import random

def show_retrieval_row(csv_file, data_folder, gt_row, other_rows, img_size=(256,256)):

    df = pd.read_csv(csv_file)

    if len(other_rows) != 10:
        raise ValueError("Provide exactly 10 satellite row indices")

    # Ground truth row
    gt_data = df.iloc[gt_row]

    ground_path = os.path.join(data_folder, gt_data["gnd_image_path"])
    sat_gt_path = os.path.join(data_folder, gt_data["sat_image_path"])

    ground_img = Image.open(ground_path).convert("RGB").resize(img_size)
    sat_gt_img = Image.open(sat_gt_path).convert("RGB").resize(img_size)

    # create figure (12 images total)
    fig, axes = plt.subplots(1, 12, figsize=(24,4))

    # Ground image
    axes[0].imshow(ground_img)
    # axes[0].set_title("Ground")
    axes[0].axis("off")

    # GT satellite
    axes[1].imshow(sat_gt_img)
    # axes[1].set_title("GT Sat")
    axes[1].axis("off")

    # other satellite images
    for i, r in enumerate(other_rows):

        sat_path = os.path.join(data_folder, df.iloc[r]["sat_image_path"])
        sat_img = Image.open(sat_path).convert("RGB").resize(img_size)

        axes[i+2].imshow(sat_img)
        # axes[i+2].set_title(f"Sat {i+1}")
        axes[i+2].axis("off")

    plt.subplots_adjust(wspace=0.02)
    # plt.tight_layout()
    # plt.savefig(f"fig/top1/{gt_row}.png")
    plt.show()



gt_row = 1842
other_rows = [random.randint(gt_row+5, 2000) for _ in range(10)]
# other_rows[4]=gt_row
print(other_rows)
# Example usage
show_retrieval_row(
    csv_file="/home/fahimul/Documents/Research/Proj_worldDataset/datasets/osv500k/test_5k.csv",
    data_folder="/home/fahimul/Documents/Research/Proj_worldDataset/datasets/osv500k/",
    gt_row=gt_row,
    other_rows=other_rows
)



## country wise 

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

# Example data
data = {
    "Metric": ["R@1","R@5","R@10"] * 5,
    # "Score": [32.42, 29.83, 29.50, 33.17, 34.5,
    #           36.67, 38.16, 45.92, 52.67, 58.67,
    #           44.83, 48.66, 59.08, 63.75, 69.0],
    "Score": [32.42, 36.67, 44.83,
              29.83, 38.16, 48.66, 
              29.50, 45.92, 59.08,
              33.17, 52.67, 63.75,
              34.5, 58.67, 69],
    "Method": ["One"]*3 + ["Two"]*3 + ["Three"]*3 + ["Four"]*3 + ["Five"]*3
}

df = pd.DataFrame(data)

sns.set_style("whitegrid")

plt.figure(figsize=(8,5))

sns.barplot(
    x="Method",
    y="Score",
    hue="Metric",      # This creates side-by-side bars
    data=df,
    palette="Set2"
)

plt.xlabel("Number of Countries", fontsize=14)
plt.ylabel("Score (%)", fontsize=14)
# plt.title("Performance Comparison on CVW500k", fontsize=16)

plt.legend(title="Metric")
plt.tight_layout()

plt.show()

## world map
I have two csv file with Latitude and Longitude of many points in the world. I want to create a world map that shows all the points. There are two variation of points, In-land and Coastal. They should be in different colors. How can I do it?  

The csv file does not have In-land and Coastal Category. The csv file have "latitude" and "longitude" column name.

In [ ]:
# !python worldmap_display.py --input datasets/osv500k/test_5k.csv --output world_map.html

!python worldmap_display.py --input datasets/osv500k/test_5k.csv --output fig/world_map.png



In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt

import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.io import shapereader


# =========================
# USER SETTINGS
# =========================
# CSV_FILE = "datasets/osv500k/train.csv"          # Change this to your CSV file path
CSV_FILE = "csv/all_v2.csv"          # Change this to your CSV file path
OUTPUT_NAME = "fig/worldmap/world_points_map_all"

LAT_COL = "latitude"
LON_COL = "longitude"

# Points within this distance from coastline will be classified as Coastal
COASTAL_THRESHOLD_KM = 56.327  # ~35 miles in km
COASTAL_THRESHOLD_mile = 35  


# Marker settings
POINT_SIZE = 12
POINT_ALPHA = 0.75


# =========================
# LOAD CSV
# =========================
df = pd.read_csv(CSV_FILE)

# Keep only valid latitude/longitude rows
df = df.dropna(subset=[LAT_COL, LON_COL]).copy()
df = df[
    (df[LAT_COL].between(-90, 90)) &
    (df[LON_COL].between(-180, 180))
].copy()

# Convert CSV points to GeoDataFrame
gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df[LON_COL], df[LAT_COL]),
    crs="EPSG:4326"
)

gdf["point_id"] = np.arange(len(gdf))


# =========================
# LOAD NATURAL EARTH DATA
# =========================
land_path = shapereader.natural_earth(
    resolution="10m",
    category="physical",
    name="land"
)

coastline_path = shapereader.natural_earth(
    resolution="10m",
    category="physical",
    name="coastline"
)

land = gpd.read_file(land_path).to_crs("EPSG:4326")
coastline = gpd.read_file(coastline_path).to_crs("EPSG:4326")


# =========================
# CHECK WHETHER POINTS ARE ON LAND
# =========================
land_join = gpd.sjoin(
    gdf[["point_id", "geometry"]],
    land[["geometry"]],
    how="left",
    predicate="within"
)

on_land_status = (
    land_join
    .groupby("point_id")["index_right"]
    .apply(lambda x: x.notna().any())
)

gdf["on_land"] = gdf["point_id"].map(on_land_status).fillna(False)


# =========================
# DISTANCE TO COASTLINE
# =========================
# Project to meters for distance calculation
# EPSG:3857 is acceptable for visualization-scale coastal classification.
gdf_m = gdf.to_crs("EPSG:3857")
coastline_m = coastline.to_crs("EPSG:3857")

nearest_coast = gpd.sjoin_nearest(
    gdf_m[["point_id", "geometry"]],
    coastline_m[["geometry"]],
    how="left",
    distance_col="distance_to_coast_m"
)

distance_to_coast = (
    nearest_coast
    .groupby("point_id")["distance_to_coast_m"]
    .min()
)

gdf["distance_to_coast_km"] = gdf["point_id"].map(distance_to_coast) / 1000


# =========================
# CLASSIFY POINTS
# =========================
gdf["category"] = np.where(
    (gdf["on_land"]) & (gdf["distance_to_coast_km"] <= COASTAL_THRESHOLD_KM),
    "Coastal",
    "In-land"
)

# Optional: if some points are not on land, label them separately
gdf.loc[gdf["on_land"] == False, "category"] = "Off-land"


# Save classified CSV
gdf.drop(columns="geometry").to_csv("classified_points.csv", index=False)


# =========================
# PLOT WORLD MAP
# =========================
fig = plt.figure(figsize=(22, 12), dpi=300)

ax = plt.axes(projection=ccrs.Robinson())
ax.set_global()

# Base map
ax.add_feature(cfeature.OCEAN, facecolor="#dceeff")
ax.add_feature(cfeature.LAND, facecolor="#f2efe9", edgecolor="none")
ax.add_feature(cfeature.COASTLINE, linewidth=0.4, edgecolor="black")
ax.add_feature(cfeature.BORDERS, linewidth=0.25, edgecolor="gray")

# Split categories
coastal = gdf[gdf["category"] == "Coastal"]
inland = gdf[gdf["category"] == "In-land"]
offland = gdf[gdf["category"] == "Off-land"]
offland = []

# Plot inland points
ax.scatter(
    inland[LON_COL],
    inland[LAT_COL],
    s=POINT_SIZE,
    c="#1f77b4",
    alpha=POINT_ALPHA,
    label="Inland",
    transform=ccrs.PlateCarree(),
    edgecolors="none"
)

# Plot coastal points
ax.scatter(
    coastal[LON_COL],
    coastal[LAT_COL],
    s=POINT_SIZE,
    c="#d62728",
    alpha=POINT_ALPHA,
    label=f"Coastal ≤ {COASTAL_THRESHOLD_mile} miles",
    transform=ccrs.PlateCarree(),
    edgecolors="none"
)

# Plot off-land points only if they exist
if len(offland) > 0:
    ax.scatter(
        offland[LON_COL],
        offland[LAT_COL],
        s=POINT_SIZE,
        c="#0C8F28",
        alpha=0.5,
        label="Off-land",
        transform=ccrs.PlateCarree(),
        edgecolors="none"
    )

# Title and legend
# plt.title(
#     "Global Distribution of In-land and Coastal Points",
#     fontsize=24,
#     fontweight="bold",
#     pad=20
# )

legend = plt.legend(
    loc="lower left",
    fontsize=14,
    frameon=True,
    facecolor="white",
    edgecolor="black"
)

# Save high-quality outputs
plt.savefig(f"{OUTPUT_NAME}.png", dpi=600, bbox_inches="tight")
plt.savefig(f"{OUTPUT_NAME}.pdf", bbox_inches="tight")
plt.savefig(f"{OUTPUT_NAME}.svg", bbox_inches="tight")

plt.show()

I need a short python script that copy a satellite image and a ground image to another folder. I have a .csv file with column name of countries, satellite image path and ground image path. Randomly select 10 countries and copy the images accordingly. rename the image file after copying. ex: {country_name}_sat.jpg, {country_name}_gnd.jpg. write a log file for tracking copied file.


In [ ]:
import os
import re
import shutil
import random
import pandas as pd
from pathlib import Path

# ======================
# Settings
# ======================
# csv_file = "datasets/osv500k/test.csv"
csv_file = "csv/all_v2.csv"
output_dir = "fig/worldmap/gnd_sat_images"
log_file = "fig/worldmap/gnd_sat_images/log.txt"
db_path = "/home/fahimul/Documents/Research/Proj_worldDataset/datasets/osv500k"

country_col = "country"
sat_col = "sat_image_path"
gnd_col = "gnd_image_path"

num_countries = 10
random_seed = 1710

# ======================
# Helper function
# ======================
def clean_name(name):
    name = str(name).strip()
    name = re.sub(r"[^\w\-]+", "_", name)
    return name.strip("_")

# ======================
# Load CSV
# ======================
df = pd.read_csv(csv_file)

# Remove rows with missing values
df = df.dropna(subset=[country_col, sat_col, gnd_col])

# Select one row per country
df_unique = df.groupby(country_col, as_index=False).first()

# Randomly select 10 countries
random.seed(random_seed)
selected_df = df_unique.sample(
    n=min(num_countries, len(df_unique)),
    random_state=random_seed
)

# Create output folder
Path(output_dir).mkdir(parents=True, exist_ok=True)

# ======================
# Copy and rename images
# ======================
with open(log_file, "a", encoding="utf-8") as log:
    log.write("country,source_sat,destination_sat,source_gnd,destination_gnd,status\n")

    for _, row in selected_df.iterrows():
        country = clean_name(row[country_col])

        sat_src = row[sat_col]
        gnd_src = row[gnd_col]

        sat_src = os.path.join(db_path, sat_src)
        gnd_src = os.path.join(db_path, gnd_src)

        sat_dst = os.path.join(output_dir, f"{country}_sat.jpg")
        gnd_dst = os.path.join(output_dir, f"{country}_gnd.jpg")

        try:
            shutil.copy2(sat_src, sat_dst)
            shutil.copy2(gnd_src, gnd_dst)

            status = "copied"

        except Exception as e:
            status = f"failed: {e}"

        log.write(
            f"{country},{sat_src},{sat_dst},{gnd_src},{gnd_dst},{status}\n"
        )

print(f"Done. Images saved in: {output_dir}")
print(f"Log saved as: {log_file}")

In [ ]:
import os
import re
import shutil
import pandas as pd
from pathlib import Path

# ======================
# Settings
# ======================
csv_file = "csv/all_v2.csv"
output_dir = "fig/worldmap/gnd_sat_images"
log_file = "fig/worldmap/gnd_sat_images/log.txt"
db_path = "/home/fahimul/Documents/Research/Proj_worldDataset/datasets/osv500k"

country_col = "country"
sat_col = "sat_image_path"
gnd_col = "gnd_image_path"

num_countries = 10
random_seed = 42

# Give country names here
selected_countries = [
    "US",
    "JP",
    "BR",
    "CN",
    "AR",
    "DE",
    "ES",
    "MX",
    "AU",
    "IT"
]

# ======================
# Helper function
# ======================
def clean_name(name):
    name = str(name).strip()
    name = re.sub(r"[^\w\-]+", "_", name)
    return name.strip("_")

# ======================
# Load CSV
# ======================
df = pd.read_csv(csv_file)

df = df.dropna(subset=[country_col, sat_col, gnd_col])

# Filter only selected countries
selected_df = df[df[country_col].isin(selected_countries)].copy()

# If multiple rows exist for the same country, keep the first one
selected_df = selected_df.groupby(country_col, as_index=False).first()

# Create output folder
Path(output_dir).mkdir(parents=True, exist_ok=True)

# ======================
# Copy and rename images
# ======================
with open(log_file, "w", encoding="utf-8") as log:
    log.write("country,source_sat,destination_sat,source_gnd,destination_gnd,status\n")

    for _, row in selected_df.iterrows():
        country = clean_name(row[country_col])

        sat_src = row[sat_col]
        gnd_src = row[gnd_col]
        
        sat_src = os.path.join(db_path, sat_src)
        gnd_src = os.path.join(db_path, gnd_src)

        sat_dst = os.path.join(output_dir, f"{country}_sat.jpg")
        gnd_dst = os.path.join(output_dir, f"{country}_gnd.jpg")

        try:
            shutil.copy2(sat_src, sat_dst)
            shutil.copy2(gnd_src, gnd_dst)

            status = "copied"

        except Exception as e:
            status = f"failed: {e}"

        log.write(
            f"{country},{sat_src},{sat_dst},{gnd_src},{gnd_dst},{status}\n"
        )

print(f"Done. Images saved in: {output_dir}")
print(f"Log saved as: {log_file}")

Great! Now I have Qualitative results for three Models of the same datasets. The models are: GeoQueryNet, GeoDTR and QDFL. Now I need to show it. Consider a samples. It can be random, but there conditions. The samples should be Top 1 correct for GeoQueryNet and incorrect for GeoDTR and QDFL. Now Create a plot with matplotlib library to display it. There will be three rows for the three models and eight column for Ground Image, Ground Truth Satellite image and Top 5 retrievals.
I will provide the dataset root path and a csv file for each samples ground and satellite location path. Use these three columns in the csv file: "id" for image ids, "gnd_image_path" for ground image path and "sat_image_path" for satellite image path. 

In [ ]:
import os
import random
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps


# ============================================================
# 1. Configuration
# ============================================================

DATASET_ROOT = "/home/fahimul/Documents/Research/Proj_worldDataset/datasets/osv500k"

DATASET_CSV = "/home/fahimul/Documents/Research/Proj_worldDataset/datasets/osv500k/test.csv"

GEOQUERYNET_RESULT_CSV = "evaluation/D10/qualitative_retrieval_results.csv"
GEODTR_RESULT_CSV = "evaluation/geodtr/qualitative_retrieval_results.csv"
QDFL_RESULT_CSV = "evaluation/qdfl/qualitative_retrieval_results.csv"

OUTPUT_DIR = "./qualitative_figures"
os.makedirs(OUTPUT_DIR, exist_ok=True)

NUM_SAMPLES_TO_PLOT = 5
RANDOM_SEED = 42

# Dataset CSV columns
ID_COL = "id"
GROUND_PATH_COL = "gnd_image_path"
SAT_PATH_COL = "sat_image_path"

# Qualitative result CSV columns
QUERY_ID_COL = "query_ground_img_id"
TRUE_SAT_ID_COL = "true_sat_img_id"
TOP1_COL = "retrieved_top1_sat_img_id"
TOP5_COL = "retrieved_top5_sat_img_ids"


# ============================================================
# 2. Utility functions
# ============================================================

def normalize_id(x):
    """
    Keeps IDs consistent even if pandas reads numbers strangely.
    """
    x = str(x).strip()
    if x.endswith(".0"):
        x = x[:-2]
    return x


def resolve_path(root, image_path):
    """
    Supports both absolute and relative paths.
    """
    image_path = str(image_path)

    if os.path.isabs(image_path):
        return image_path

    return os.path.join(root, image_path)


def load_image(image_path, image_size=224):
    """
    Loads and center-crops image for clean visualization.
    """
    try:
        img = Image.open(image_path).convert("RGB")
        img = ImageOps.fit(img, (image_size, image_size), method=Image.Resampling.LANCZOS)
        return img
    except Exception as e:
        print(f"Could not load image: {image_path}")
        print(e)
        return Image.new("RGB", (image_size, image_size), color=(240, 240, 240))


def split_topk_ids(topk_string):
    """
    Handles Top-5 IDs saved as:
        id1|id2|id3|id4|id5
    Also supports comma-separated format.
    """
    if pd.isna(topk_string):
        return []

    topk_string = str(topk_string).strip()

    if "|" in topk_string:
        ids = topk_string.split("|")
    elif "," in topk_string:
        ids = topk_string.split(",")
    else:
        ids = topk_string.split()

    return [normalize_id(x) for x in ids if str(x).strip() != ""]


def is_top1_correct(row):
    return normalize_id(row[TRUE_SAT_ID_COL]) == normalize_id(row[TOP1_COL])


# ============================================================
# 3. Load dataset metadata
# ============================================================

dataset_df = pd.read_csv(DATASET_CSV, dtype=str)

dataset_df[ID_COL] = dataset_df[ID_COL].apply(normalize_id)

id_to_ground_path = {}
id_to_sat_path = {}

for _, row in dataset_df.iterrows():
    img_id = normalize_id(row[ID_COL])

    id_to_ground_path[img_id] = resolve_path(
        DATASET_ROOT,
        row[GROUND_PATH_COL]
    )

    id_to_sat_path[img_id] = resolve_path(
        DATASET_ROOT,
        row[SAT_PATH_COL]
    )


# ============================================================
# 4. Load qualitative result CSVs
# ============================================================

def load_result_csv(csv_path, model_name):
    df = pd.read_csv(csv_path, dtype=str)

    df[QUERY_ID_COL] = df[QUERY_ID_COL].apply(normalize_id)
    df[TRUE_SAT_ID_COL] = df[TRUE_SAT_ID_COL].apply(normalize_id)
    df[TOP1_COL] = df[TOP1_COL].apply(normalize_id)

    df["top1_correct_computed"] = df.apply(is_top1_correct, axis=1)
    df["model"] = model_name

    return df


geoquerynet_df = load_result_csv(
    GEOQUERYNET_RESULT_CSV,
    "GeoQueryNet"
)

geodtr_df = load_result_csv(
    GEODTR_RESULT_CSV,
    "GeoDTR"
)

qdfl_df = load_result_csv(
    QDFL_RESULT_CSV,
    "QDFL"
)
# print(geoquerynet_df.loc[geoquerynet_df["top1_correct_computed"] == True].shape)

# ============================================================
# 5. Find samples satisfying condition
# ============================================================

geoquerynet_good_ids = set(
    geoquerynet_df.loc[
        geoquerynet_df["top1_correct_computed"] == True,
        QUERY_ID_COL
    ]
)
print(len(geoquerynet_good_ids))

geodtr_bad_ids = set(
    geodtr_df.loc[
        geodtr_df["top1_correct_computed"] == False,
        QUERY_ID_COL
    ]
)
print(len(geodtr_bad_ids))

qdfl_bad_ids = set(
    qdfl_df.loc[
        qdfl_df["top1_correct_computed"] == False,
        QUERY_ID_COL
    ]
)
print(len(qdfl_bad_ids))

candidate_ids = sorted(
    geoquerynet_good_ids
    & geodtr_bad_ids
    & qdfl_bad_ids
)
print(sorted(geoquerynet_good_ids))
print(sorted(geodtr_bad_ids))
print(sorted(qdfl_bad_ids))


print(f"Number of candidate samples found: {len(candidate_ids)}")

if len(candidate_ids) == 0:
    raise ValueError(
        "No sample found where GeoQueryNet is Top-1 correct "
        "and both GeoDTR and QDFL are Top-1 incorrect."
    )

random.seed(RANDOM_SEED)
selected_ids = random.sample(
    candidate_ids,
    k=min(NUM_SAMPLES_TO_PLOT, len(candidate_ids))
)

print("Selected sample IDs:", selected_ids)


# Save selected IDs for record
pd.DataFrame({"selected_id": selected_ids}).to_csv(
    os.path.join(OUTPUT_DIR, "selected_qualitative_sample_ids.csv"),
    index=False
)


# ============================================================
# 6. Convert result dataframe into dictionary for quick access
# ============================================================

geoquerynet_dict = {
    normalize_id(row[QUERY_ID_COL]): row
    for _, row in geoquerynet_df.iterrows()
}

geodtr_dict = {
    normalize_id(row[QUERY_ID_COL]): row
    for _, row in geodtr_df.iterrows()
}

qdfl_dict = {
    normalize_id(row[QUERY_ID_COL]): row
    for _, row in qdfl_df.iterrows()
}

model_results = {
    "GeoQueryNet": geoquerynet_dict,
    "GeoDTR": geodtr_dict,
    "QDFL": qdfl_dict,
}


# ============================================================
# 7. Plot one qualitative comparison figure
# ============================================================

def plot_qualitative_sample(sample_id, save_path):
    model_names = ["GeoQueryNet", "GeoDTR", "QDFL"]

    column_titles = [
        "Model",
        "Ground",
        "GT Satellite",
        "Top-1",
        "Top-2",
        "Top-3",
        "Top-4",
        "Top-5",
    ]

    fig, axes = plt.subplots(
        nrows=3,
        ncols=8,
        figsize=(24, 9),
        dpi=300
    )

    fig.suptitle(
        f"Qualitative Cross-View Retrieval Comparison | Query ID: {sample_id}",
        fontsize=18,
        fontweight="bold",
        y=0.98
    )

    for col_idx, title in enumerate(column_titles):
        axes[0, col_idx].set_title(title, fontsize=13, fontweight="bold")

    for row_idx, model_name in enumerate(model_names):
        result_row = model_results[model_name][sample_id]

        query_id = normalize_id(result_row[QUERY_ID_COL])
        true_sat_id = normalize_id(result_row[TRUE_SAT_ID_COL])
        top5_ids = split_topk_ids(result_row[TOP5_COL])

        # Make sure we have exactly 5 IDs for plotting
        top5_ids = top5_ids[:5]
        while len(top5_ids) < 5:
            top5_ids.append(None)

        # ----------------------------------------------------
        # Column 0: Model name
        # ----------------------------------------------------
        ax = axes[row_idx, 0]
        ax.axis("off")

        top1_correct = normalize_id(top5_ids[0]) == true_sat_id

        if top1_correct:
            status = "Top-1 Correct"
            status_color = "green"
        else:
            status = "Top-1 Incorrect"
            status_color = "red"

        ax.text(
            0.5,
            0.58,
            model_name,
            ha="center",
            va="center",
            fontsize=15,
            fontweight="bold"
        )

        ax.text(
            0.5,
            0.38,
            status,
            ha="center",
            va="center",
            fontsize=11,
            color=status_color,
            fontweight="bold"
        )

        # ----------------------------------------------------
        # Column 1: Ground image
        # ----------------------------------------------------
        ground_path = id_to_ground_path[query_id]
        ground_img = load_image(ground_path)

        ax = axes[row_idx, 1]
        ax.imshow(ground_img)
        ax.axis("off")

        # ----------------------------------------------------
        # Column 2: Ground-truth satellite
        # ----------------------------------------------------
        gt_sat_path = id_to_sat_path[true_sat_id]
        gt_sat_img = load_image(gt_sat_path)

        ax = axes[row_idx, 2]
        ax.imshow(gt_sat_img)
        ax.axis("off")

        for spine in ax.spines.values():
            spine.set_edgecolor("green")
            spine.set_linewidth(4)

        # ----------------------------------------------------
        # Columns 3-7: Top-5 retrieved satellite images
        # ----------------------------------------------------
        for k in range(5):
            retrieved_id = top5_ids[k]
            ax = axes[row_idx, k + 3]

            if retrieved_id is None or retrieved_id not in id_to_sat_path:
                img = Image.new("RGB", (224, 224), color=(240, 240, 240))
                ax.imshow(img)
                ax.set_title("Missing", fontsize=9)
                ax.axis("off")
                continue

            retrieved_sat_path = id_to_sat_path[retrieved_id]
            retrieved_img = load_image(retrieved_sat_path)

            ax.imshow(retrieved_img)
            ax.axis("off")

            if retrieved_id == true_sat_id:
                border_color = "green"
                title = f"ID: {retrieved_id}\nCorrect"
            else:
                border_color = "red"
                title = f"ID: {retrieved_id}"

            ax.set_title(title, fontsize=8)

            for spine in ax.spines.values():
                spine.set_visible(True)
                spine.set_edgecolor(border_color)
                spine.set_linewidth(4)

    plt.tight_layout(rect=[0, 0, 1, 0.94])
    plt.savefig(save_path, bbox_inches="tight", dpi=300)
    plt.close()

    print(f"Saved figure: {save_path}")


# ============================================================
# 8. Generate figures
# ============================================================

for sample_id in selected_ids:
    save_path = os.path.join(
        OUTPUT_DIR,
        f"qualitative_comparison_{sample_id}.png"
    )

    plot_qualitative_sample(
        sample_id=sample_id,
        save_path=save_path
    )

In [5]:
import os
import random
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps


# ============================================================
# 1. Configuration
# ============================================================

DATASET_ROOT = "/home/fahimul/Documents/Research/Proj_worldDataset/datasets/osv500k"

DATASET_CSV = "/home/fahimul/Documents/Research/Proj_worldDataset/datasets/osv500k/test.csv"

GEOQUERYNET_RESULT_CSV = "evaluation/D10/qualitative_retrieval_results.csv"
GEODTR_RESULT_CSV = "evaluation/geodtr/qualitative_retrieval_results.csv"
QDFL_RESULT_CSV = "evaluation/qdfl/qualitative_retrieval_results.csv"

OUTPUT_DIR = "./qualitative_figures"
os.makedirs(OUTPUT_DIR, exist_ok=True)

NUM_SAMPLES_TO_PLOT = 5
RANDOM_SEED = 24

# Dataset CSV columns
ID_COL = "id"
GROUND_PATH_COL = "gnd_image_path"
SAT_PATH_COL = "sat_image_path"

# Result CSV columns
QUERY_ROW_COL = "query_row_index"
TOP5_COL = "retrieved_top5_sat_img_ids"


# DATASET_ROOT = "/home/fahimul/Documents/Research/Proj_worldDataset/datasets/osv500k"

# DATASET_CSV = "/home/fahimul/Documents/Research/Proj_worldDataset/datasets/osv500k/test.csv"

# GEOQUERYNET_RESULT_CSV = "evaluation/D10/qualitative_retrieval_results.csv"
# GEODTR_RESULT_CSV = "evaluation/geodtr/qualitative_retrieval_results.csv"
# QDFL_RESULT_CSV = "evaluation/qdfl/qualitative_retrieval_results.csv"

# OUTPUT_DIR = "./qualitative_figures"
# os.makedirs(OUTPUT_DIR, exist_ok=True)

# NUM_SAMPLES_TO_PLOT = 5
# RANDOM_SEED = 42

# # Dataset CSV columns
# ID_COL = "id"
# GROUND_PATH_COL = "gnd_image_path"
# SAT_PATH_COL = "sat_image_path"

# # Qualitative result CSV columns
# QUERY_ID_COL = "query_ground_img_id"
# TRUE_SAT_ID_COL = "true_sat_img_id"
# TOP1_COL = "retrieved_top1_sat_img_id"
# TOP5_COL = "retrieved_top5_sat_img_ids"


# ============================================================
# 2. Utility functions
# ============================================================

def normalize_row_index(x):
    """
    Converts row index values like '15', '15.0', 15 into int.
    """
    if pd.isna(x):
        return None

    x = str(x).strip()

    if x.endswith(".0"):
        x = x[:-2]

    return int(x)


def split_top5_row_indices(top5_string):
    """
    Top-5 IDs are saved as:
        index_row_id1|index_row_id2|index_row_id3|index_row_id4|index_row_id5
    """
    if pd.isna(top5_string):
        return []

    top5_string = str(top5_string).strip()

    if "|" in top5_string:
        parts = top5_string.split("|")
    elif "," in top5_string:
        parts = top5_string.split(",")
    else:
        parts = top5_string.split()

    row_indices = []

    for p in parts:
        p = p.strip()
        if p != "":
            row_indices.append(normalize_row_index(p))

    return row_indices


def resolve_path(root, image_path):
    """
    Supports both absolute and relative image paths.
    """
    image_path = str(image_path)

    if os.path.isabs(image_path):
        return image_path

    return os.path.join(root, image_path)


def load_image(image_path, image_size=224):
    """
    Load image and resize/crop for clean plotting.
    """
    try:
        img = Image.open(image_path).convert("RGB")
        img = ImageOps.fit(
            img,
            (image_size, image_size),
            method=Image.Resampling.LANCZOS
        )
        return img

    except Exception as e:
        print(f"Could not load image: {image_path}")
        print(e)
        return Image.new("RGB", (image_size, image_size), color=(240, 240, 240))


def get_ground_path_from_row(dataset_df, row_index):
    row = dataset_df.iloc[row_index]
    return resolve_path(DATASET_ROOT, row[GROUND_PATH_COL])


def get_sat_path_from_row(dataset_df, row_index):
    row = dataset_df.iloc[row_index]
    return resolve_path(DATASET_ROOT, row[SAT_PATH_COL])


def get_display_id_from_row(dataset_df, row_index):
    """
    This is only for writing titles.
    The retrieval matching itself uses row index.
    """
    if ID_COL in dataset_df.columns:
        return str(dataset_df.iloc[row_index][ID_COL])
    return str(row_index)


def show_image(ax, img, title=None, border_color=None, linewidth=4):
    ax.imshow(img)
    ax.set_xticks([])
    ax.set_yticks([])

    if title is not None:
        ax.set_title(title, fontsize=8)

    if border_color is not None:
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_edgecolor(border_color)
            spine.set_linewidth(linewidth)
    else:
        for spine in ax.spines.values():
            spine.set_visible(False)


# ============================================================
# 3. Load dataset CSV
# ============================================================

dataset_df = pd.read_csv(DATASET_CSV)

# Very important:
# keep the row order exactly the same as evaluation.
dataset_df = dataset_df.reset_index(drop=True)

print(f"Loaded dataset CSV with {len(dataset_df)} rows")


# ============================================================
# 4. Load result CSVs
# ============================================================

def load_result_csv(csv_path, model_name):
    df = pd.read_csv(csv_path)

    df[QUERY_ROW_COL] = df[QUERY_ROW_COL].apply(normalize_row_index)

    df["top5_row_indices"] = df[TOP5_COL].apply(split_top5_row_indices)

    df["retrieved_top1_row_index"] = df["top5_row_indices"].apply(
        lambda x: x[0] if len(x) > 0 else None
    )

    # Because ground and satellite positive pair are from the same row
    df["true_sat_row_index"] = df[QUERY_ROW_COL]

    df["top1_correct_computed"] = (
        df["retrieved_top1_row_index"] == df["true_sat_row_index"]
    )

    df["model"] = model_name

    return df


geoquerynet_df = load_result_csv(
    GEOQUERYNET_RESULT_CSV,
    "GeoQueryNet"
)

geodtr_df = load_result_csv(
    GEODTR_RESULT_CSV,
    "GeoDTR"
)

qdfl_df = load_result_csv(
    QDFL_RESULT_CSV,
    "QDFL"
)


# ============================================================
# 5. Find valid samples
#    Condition:
#    GeoQueryNet Top-1 correct
#    GeoDTR Top-1 incorrect
#    QDFL Top-1 incorrect
# ============================================================

geoquerynet_good_rows = set(
    geoquerynet_df.loc[
        geoquerynet_df["top1_correct_computed"] == True,
        QUERY_ROW_COL
    ]
)

geodtr_bad_rows = set(
    geodtr_df.loc[
        geodtr_df["top1_correct_computed"] == False,
        QUERY_ROW_COL
    ]
)

qdfl_bad_rows = set(
    qdfl_df.loc[
        qdfl_df["top1_correct_computed"] == False,
        QUERY_ROW_COL
    ]
)

candidate_rows = sorted(
    geoquerynet_good_rows
    & geodtr_bad_rows
    & qdfl_bad_rows
)

print(f"Number of candidate samples found: {len(candidate_rows)}")

if len(candidate_rows) == 0:
    raise ValueError(
        "No sample found where GeoQueryNet is Top-1 correct "
        "and both GeoDTR and QDFL are Top-1 incorrect."
    )

random.seed(RANDOM_SEED)

selected_rows = random.sample(
    candidate_rows,
    k=min(NUM_SAMPLES_TO_PLOT, len(candidate_rows))
)

print("Selected dataset row indices:", selected_rows)

pd.DataFrame({"selected_row_index": selected_rows}).to_csv(
    os.path.join(OUTPUT_DIR, "selected_qualitative_row_indices.csv"),
    index=False
)


# ============================================================
# 6. Convert each result dataframe into dictionary
# ============================================================

def df_to_row_dict(df):
    return {
        int(row[QUERY_ROW_COL]): row
        for _, row in df.iterrows()
    }


geoquerynet_dict = df_to_row_dict(geoquerynet_df)
geodtr_dict = df_to_row_dict(geodtr_df)
qdfl_dict = df_to_row_dict(qdfl_df)

model_results = {
    "GeoQueryNet": geoquerynet_dict,
    "GeoDTR": geodtr_dict,
    "QDFL": qdfl_dict,
}


# ============================================================
# 7. Plot qualitative comparison
# ============================================================

def plot_qualitative_sample(query_row_index, save_path):
    model_names = ["GeoQueryNet", "GeoDTR", "QDFL"]

    column_titles = [
        "Model",
        "Ground",
        "GT Satellite",
        "Top-1",
        "Top-2",
        "Top-3",
        "Top-4",
        "Top-5",
    ]

    fig, axes = plt.subplots(
        nrows=3,
        ncols=8,
        figsize=(24, 9),
        dpi=300
    )

    query_display_id = get_display_id_from_row(dataset_df, query_row_index)

    fig.suptitle(
        f"Qualitative Retrieval Comparison | Query Row: {query_row_index}, ID: {query_display_id}",
        fontsize=18,
        fontweight="bold",
        y=0.98
    )

    for col_idx, title in enumerate(column_titles):
        axes[0, col_idx].set_title(title, fontsize=13, fontweight="bold")

    # Since anchor and positive are paired in the same dataset row
    true_sat_row_index = query_row_index

    ground_path = get_ground_path_from_row(dataset_df, query_row_index)
    gt_sat_path = get_sat_path_from_row(dataset_df, true_sat_row_index)

    ground_img = load_image(ground_path)
    gt_sat_img = load_image(gt_sat_path)

    for row_idx, model_name in enumerate(model_names):
        result_row = model_results[model_name][query_row_index]

        top5_row_indices = result_row["top5_row_indices"]

        top5_row_indices = top5_row_indices[:5]
        while len(top5_row_indices) < 5:
            top5_row_indices.append(None)

        top1_row_index = top5_row_indices[0]
        top1_correct = top1_row_index == true_sat_row_index

        # ----------------------------------------------------
        # Column 0: Model name
        # ----------------------------------------------------
        ax = axes[row_idx, 0]
        ax.set_xticks([])
        ax.set_yticks([])

        for spine in ax.spines.values():
            spine.set_visible(False)

        if top1_correct:
            status = "Top-1 Correct"
            status_color = "green"
        else:
            status = "Top-1 Incorrect"
            status_color = "red"

        ax.text(
            0.5,
            0.58,
            model_name,
            ha="center",
            va="center",
            fontsize=15,
            fontweight="bold"
        )

        ax.text(
            0.5,
            0.38,
            status,
            ha="center",
            va="center",
            fontsize=11,
            color=status_color,
            fontweight="bold"
        )

        # ----------------------------------------------------
        # Column 1: Ground image
        # ----------------------------------------------------
        show_image(
            axes[row_idx, 1],
            ground_img,
            title=f"Row: {query_row_index}\nID: {query_display_id}",
            border_color=None
        )

        # ----------------------------------------------------
        # Column 2: Ground-truth satellite image
        # ----------------------------------------------------
        show_image(
            axes[row_idx, 2],
            gt_sat_img,
            title=f"GT Row: {true_sat_row_index}",
            border_color="green"
        )

        # ----------------------------------------------------
        # Columns 3-7: Top-5 retrieved satellite images
        # ----------------------------------------------------
        for k in range(5):
            retrieved_row_index = top5_row_indices[k]
            ax = axes[row_idx, k + 3]

            if retrieved_row_index is None:
                blank = Image.new("RGB", (224, 224), color=(240, 240, 240))
                show_image(ax, blank, title="Missing", border_color=None)
                continue

            retrieved_sat_path = get_sat_path_from_row(
                dataset_df,
                retrieved_row_index
            )

            retrieved_img = load_image(retrieved_sat_path)

            retrieved_display_id = get_display_id_from_row(
                dataset_df,
                retrieved_row_index
            )

            if retrieved_row_index == true_sat_row_index:
                border_color = "green"
                title = f"Row: {retrieved_row_index}\nID: {retrieved_display_id}\nCorrect"
            else:
                border_color = "red"
                title = f"Row: {retrieved_row_index}\nID: {retrieved_display_id}"

            show_image(
                ax,
                retrieved_img,
                title=title,
                border_color=border_color
            )

    plt.tight_layout(rect=[0, 0, 1, 0.94])
    plt.savefig(save_path, bbox_inches="tight", dpi=300)
    plt.close()

    print(f"Saved figure: {save_path}")


# ============================================================
# 8. Generate figures
# ============================================================

for query_row_index in selected_rows:
    save_path = os.path.join(
        OUTPUT_DIR,
        f"qualitative_comparison_row_{query_row_index}.png"
    )

    plot_qualitative_sample(
        query_row_index=query_row_index,
        save_path=save_path
    )

Loaded dataset CSV with 48735 rows
Number of candidate samples found: 2402
Selected dataset row indices: [30785, 48448, 15175, 18291, 14033]
Saved figure: ./qualitative_figures/qualitative_comparison_row_30785.png
Saved figure: ./qualitative_figures/qualitative_comparison_row_48448.png
Saved figure: ./qualitative_figures/qualitative_comparison_row_15175.png
Saved figure: ./qualitative_figures/qualitative_comparison_row_18291.png
Saved figure: ./qualitative_figures/qualitative_comparison_row_14033.png
